In [1]:
import sys
import os
from pathlib import Path
import random

# Add source directory to path
sys.path.append(os.path.abspath("../src"))

# Import project utilities
from abstractionssymh.debug_utils import debug_info, debug_error, debug_success
from abstractionssymh.data_loader import parse_json_to_dsl
from abstractionssymh.plot_utils import plot_dsl_with_k3d
from abstractionssymh.dsl_utils import collect_singleton_and_pair_data
from abstractionssymh.abstraction_utils import find_abstractions, Abstraction, integrate_abstractions

In [2]:
def get_dataset_directory():
    current_path = Path.cwd()
    base_project_dir = current_path.parent
    dataset_directory = base_project_dir / "src" / "abstractionssymh" / "dataset"

    debug_info("Current notebook location:", current_path)
    debug_info("Base project directory:", base_project_dir)
    debug_info("Target dataset directory:", dataset_directory)

    return base_project_dir, dataset_directory

base_project_dir, dataset_directory = get_dataset_directory()

[INFO] Current notebook location: /Users/amogh/Projects/abstraction-discovery/notebookssymh
[INFO] Base project directory: /Users/amogh/Projects/abstraction-discovery
[INFO] Target dataset directory: /Users/amogh/Projects/abstraction-discovery/src/abstractionssymh/dataset


In [3]:
def load_chair_dsl(chair_directory, use_random=False):
    json_files = sorted(chair_directory.glob("*.json"))
    if not json_files:
        debug_error("No JSON files found in:", chair_directory)
        return None

    file_path = random.choice(json_files) if use_random else json_files[0]
    debug_success("Loading chair file:", file_path)

    json_content = file_path.read_text(encoding="utf-8")
    dsl_object = parse_json_to_dsl(json_content)

    debug_success("Successfully parsed DSL object")
    debug_info(dsl_object)
    return dsl_object

def plot_chair(dsl_obj):
    if dsl_obj is None:
        debug_error("No DSL object to plot.")
        return
    try:
        debug_info("Rendering DSL object with k3d...")
        plot_dsl_with_k3d(dsl_obj)
        debug_success("Plotting complete.")
    except Exception as e:
        debug_error("Failed to plot DSL object:", e)

In [4]:
chair_directory = dataset_directory / "Chair"
dsl_object = load_chair_dsl(chair_directory, use_random=False)
plot_chair(dsl_object)

[SUCCESS] Loading chair file: /Users/amogh/Projects/abstraction-discovery/src/abstractionssymh/dataset/Chair/Chair_1.json
[SUCCESS] Successfully parsed DSL object
[INFO] Union(
    Union(
        Translate(center=[0.003, 0.149, 0.072])
            Rotate(quat=[0.0000, 0.0000, 0.0000, 1.0000])
                Scale(lengths=[0.891, 0.224, 0.806])
                    Box(label=1),
        Translate(center=[0.004, -0.422, 0.170])
            Rotate(quat=[-0.3389, -0.0000, 0.0065, 0.9408])
                Scale(lengths=[0.898, 0.524, 1.154])
                    Box(label=2)
    ),
    Translate(center=[0.030, 0.488, -0.315])
        Rotate(quat=[-0.0821, -0.0071, -0.0494, 0.9954])
            Scale(lengths=[0.892, 0.781, 0.305])
                Box(label=0)
)
[INFO] Rendering DSL object with k3d...
[INFO] Expanding DSL tree for visualization...
[INFO] Found 3 total boxes after expansion.


Output()

[SUCCESS] 3D plot displayed successfully.
[SUCCESS] Plotting complete.


In [5]:
import pickle
from pathlib import Path

saved_directory = base_project_dir / "src" / "abstractionssymh" / "saved"
saved_directory.mkdir(parents=True, exist_ok=True)

# File paths for pickled data
singleton_file = saved_directory / Path("singleton_params.pkl")
pair_file = saved_directory / Path("pair_params.pkl")

In [ ]:
# if singleton_file.is_file() and pair_file.is_file():
#     with open(singleton_file, "rb") as f:
#         singleton_params = pickle.load(f)
#     with open(pair_file, "rb") as f:
#         pair_params = pickle.load(f)
#     debug_success("Loaded singleton and pair parameters from files.")

#     num_singleton = sum(len(v) for v in singleton_params.values())
#     num_pair = sum(len(v) for v in pair_params.values())
#     debug_success(f"Loaded {num_singleton} singleton and {num_pair} pair parameter sets.")
# else:
#     debug_info("Pickle files not found. Uncomment the collection block to generate new data.")

In [6]:
chair_directory = dataset_directory / "Chair"
json_files = sorted(list(chair_directory.glob("*.json")))

try:
    debug_info("Starting to load DSL shapes from JSON files...")
    all_dsl_shapes = [parse_json_to_dsl(Path(f).read_text()) for f in json_files[:500]]
    # all_dsl_shapes = [parse_json_to_dsl(Path(f).read_text()) for f in json_files]
    debug_success(f"Loaded {len(all_dsl_shapes)} shapes from dataset.")
except Exception as e:
    debug_error("Failed to load DSL shapes:", e)
    all_dsl_shapes = []

[INFO] Starting to load DSL shapes from JSON files...
[SUCCESS] Loaded 500 shapes from dataset.


In [7]:
# Collect data fresh
singleton_params, pair_params = collect_singleton_and_pair_data(all_dsl_shapes)

# Save to files
with open(singleton_file, "wb") as f:
    pickle.dump(singleton_params, f)
with open(pair_file, "wb") as f:
    pickle.dump(pair_params, f)
debug_success("Collected and saved new singleton and pair parameter sets.")

num_singleton = sum(len(v) for v in singleton_params.values())
num_pair = sum(len(v) for v in pair_params.values())
debug_success(f"Collected {num_singleton} singleton and {num_pair} pair parameter sets.")

[SUCCESS] Collected keys: dict_keys(['Scale', 'Rotate', 'Translate', 'SymRef', 'SymRot', 'SymTrans']) singletons, dict_keys(['Scale(Box)', 'Rotate(Scale)', 'Translate(Rotate)', 'Union(Translate)', 'SymRef(Union)', 'Union(SymRef)', 'SymRef(Translate)', 'SymRot(Union)', 'Union(SymRot)', 'SymRot(Translate)', 'SymTrans(Translate)', 'Union(SymTrans)']) pairs
[SUCCESS] Collected parameters: 6 singletons, 12 pairs
[SUCCESS] Collected and saved new singleton and pair parameter sets.
[SUCCESS] Collected 10888 singleton and 14759 pair parameter sets.


In [8]:
import numpy as np

for key in singleton_params.keys():
    print(f"{key}: {len(singleton_params[key])} samples, with size {np.array(singleton_params[key][0]).size}")

for key in pair_params.keys():
    print(f"{key}: {len(pair_params[key])} samples, with size {np.array(pair_params[key][0]).size}")

Rotate: 3287 samples, with size 4
Scale: 3287 samples, with size 3
SymRef: 1001 samples, with size 6
SymRot: 9 samples, with size 6
SymTrans: 17 samples, with size 3
Translate: 3287 samples, with size 3
Rotate(Scale): 3287 samples, with size 7
Scale(Box): 3287 samples, with size 3
SymRef(Translate): 424 samples, with size 9
SymRef(Union): 577 samples, with size 6
SymRot(Translate): 2 samples, with size 9
SymRot(Union): 7 samples, with size 6
SymTrans(Translate): 17 samples, with size 6
Translate(Rotate): 3287 samples, with size 7
Union(SymRef): 1001 samples, with size 6
Union(SymRot): 9 samples, with size 6
Union(SymTrans): 17 samples, with size 3
Union(Translate): 2844 samples, with size 3


In [9]:
import os
import numpy as np
import pandas as pd
import hdbscan
import umap
import k3d
import seaborn as sns
import matplotlib.pyplot as plt

# --- 2. Configuration ---
# Define meaningful column names for the pandas output
COLUMN_NAMES = {
    'Rotate': ['quat_w', 'quat_x', 'quat_y', 'quat_z'],
    'Scale': ['scale_x', 'scale_y', 'scale_z'],
    'SymRef': ['plane_pt_x', 'plane_pt_y', 'plane_pt_z', 'normal_x', 'normal_y', 'normal_z'],
    'SymRot': ['axis_pt_x', 'axis_pt_y', 'axis_pt_z', 'axis_dir_x', 'axis_dir_y', 'axis_dir_z'],
    'SymTrans': ['vec_x', 'vec_y', 'vec_z'],
    'Translate': ['trans_x', 'trans_y', 'trans_z'],
}

# Create a directory to save the plots
output_dir = "singleton_cluster_plots"
os.makedirs(output_dir, exist_ok=True)


# --- 3. Main Analysis Loop ---
for name, data in singleton_params.items():
    data_array = np.array(data, dtype=np.float32)
    n_samples, n_dims = data_array.shape

    print(f"\n{'='*70}")
    print(f"ANALYZING: {name} ({n_samples} samples, {n_dims} dimensions)")
    print(f"{'='*70}")

    if n_samples < 50:
        print("Skipping: Not enough samples for meaningful clustering.")
        continue

    # === STEP 1: ROBUST CLUSTERING (HDBSCAN) ===
    # Set min_cluster_size to 0.5% of samples, with a floor of 50
    min_cluster_size = max(50, int(n_samples * 0.005))
    clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size,
                                gen_min_span_tree=True)
    
    print(f"Running HDBSCAN with min_cluster_size = {min_cluster_size}...")
    labels = clusterer.fit_predict(data_array)
    n_clusters = labels.max() + 1
    n_noise = np.count_nonzero(labels == -1)
    print(f"-> Found {n_clusters} clusters and {n_noise} noise points.")

    if n_clusters == 0:
        print("-> No distinct clusters found. Skipping further analysis.")
        continue

    # === STEP 2: STATISTICAL PROFILING (PANDAS) ===
    print("\n--- Statistical Profile of Clusters ---")
    df = pd.DataFrame(data_array, columns=COLUMN_NAMES.get(name, [f'param_{i}' for i in range(n_dims)]))
    df['cluster'] = labels
    
    # Exclude noise points for profiling
    profile = df[df['cluster'] != -1].groupby('cluster').agg(['mean', 'std', 'count'])
    pd.set_option('display.max_columns', None)
    print(profile)
    print("Interpretation Guide:")
    print("- 'mean': The average parameter values for this cluster.")
    print("- 'std': Low values indicate a defining characteristic (all points are similar).")
    print("- 'count': The number of points in the cluster.")


    # === STEP 3: VISUAL ANALYSIS (UMAP & K3D) ===
    print("\n--- Generating 3D Visualization ---")
    plot_positions = data_array
    plot_axes = ['X', 'Y', 'Z'] # Default for 3D data

    if n_dims > 3:
        print(f"Reducing {n_dims}D data to 3D with UMAP...")
        reducer = umap.UMAP(n_components=3, n_neighbors=30, min_dist=0.1, random_state=42)
        plot_positions = reducer.fit_transform(data_array)
        plot_axes = ['UMAP 1', 'UMAP 2', 'UMAP 3']

    # Create color map
    color_palette = sns.color_palette("husl", n_clusters).as_hex()
    color_map = {i: int(c.lstrip('#'), 16) for i, c in enumerate(color_palette)}
    
    point_colors = np.full(n_samples, 0x808080, dtype=np.uint32) # Noise is grey
    for i, label in enumerate(labels):
        if label != -1:
            point_colors[i] = color_map[label]

    # Create and save K3D plot
    plot = k3d.plot(name=f'Clusters for {name}')
    plot += k3d.points(positions=plot_positions.astype(np.float32),
                       colors=point_colors,
                       point_size=0.05,
                       shader='3d')
    plot.axes = plot_axes
    
    html_filename = os.path.join(output_dir, f"{name}_clusters.html")
    try:
        with open(html_filename, "w", encoding="utf-8") as f:
            f.write(plot.get_snapshot())
        print(f"-> Successfully saved interactive plot to: {html_filename}")
    except Exception as e:
        print(f"-> Could not save plot. Error: {e}")


print(f"\n{'='*70}")
print("Analysis complete. Check the console for statistical profiles and the '{output_dir}' directory for interactive plots.")



ANALYZING: Rotate (3287 samples, 4 dimensions)
Running HDBSCAN with min_cluster_size = 50...
-> Found 2 clusters and 1083 noise points.

--- Statistical Profile of Clusters ---
           quat_w                    quat_x                    quat_y  \
             mean       std count      mean       std count      mean   
cluster                                                                 
0       -0.109968  0.008729    57 -0.000307  0.002021    57  0.000180   
1       -0.002977  0.012820  2147 -0.000061  0.000710  2147  0.000004   

                           quat_z                 
              std count      mean      std count  
cluster                                           
0        0.001895    57  0.993893  0.00093    57  
1        0.001268  2147  0.999912  0.00043  2147  
Interpretation Guide:
- 'mean': The average parameter values for this cluster.
- 'std': Low values indicate a defining characteristic (all points are similar).
- 'count': The number of points in the cl

/opt/homebrew/Caskroom/miniconda/base/envs/abs/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/abs/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/abs/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


-> Successfully saved interactive plot to: singleton_cluster_plots/Rotate_clusters.html

ANALYZING: Scale (3287 samples, 3 dimensions)
Running HDBSCAN with min_cluster_size = 50...
-> Found 6 clusters and 939 noise points.

--- Statistical Profile of Clusters ---
          scale_x                   scale_y                   scale_z  \
             mean       std count      mean       std count      mean   
cluster                                                                 
0        0.828187  0.125453   416  0.202348  0.117900   416  0.795374   
1        0.077881  0.031271   246  0.074757  0.029318   246  0.671569   
2        0.721417  0.089133   230  0.113540  0.121571   230  0.085093   
3        0.756799  0.086609   123  0.782694  0.079790   123  0.143930   
4        0.072958  0.030031  1160  0.618149  0.173620  1160  0.085049   
5        0.085595  0.049819   173  0.078579  0.053409   173  0.078724   

                         
              std count  
cluster                  


/opt/homebrew/Caskroom/miniconda/base/envs/abs/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/abs/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


-> Successfully saved interactive plot to: singleton_cluster_plots/Scale_clusters.html

ANALYZING: SymRef (1001 samples, 6 dimensions)
Running HDBSCAN with min_cluster_size = 50...
-> Found 4 clusters and 374 noise points.

--- Statistical Profile of Clusters ---
        plane_pt_x                 plane_pt_y                 plane_pt_z  \
              mean       std count       mean       std count       mean   
cluster                                                                    
0         0.999677  0.000999   234   0.000110  0.010191   234  -0.002891   
1         0.999804  0.000711   162   0.000473  0.011037   162   0.001328   
2         0.999803  0.000383    80   0.003724  0.011497    80   0.005715   
3         0.999863  0.000283   151   0.000354  0.009182   151  -0.000856   

                         normal_x                  normal_y                  \
              std count      mean       std count      mean       std count   
cluster                                      

/opt/homebrew/Caskroom/miniconda/base/envs/abs/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/abs/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/abs/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


-> Successfully saved interactive plot to: singleton_cluster_plots/SymRef_clusters.html

ANALYZING: SymRot (9 samples, 6 dimensions)
Skipping: Not enough samples for meaningful clustering.

ANALYZING: SymTrans (17 samples, 3 dimensions)
Skipping: Not enough samples for meaningful clustering.

ANALYZING: Translate (3287 samples, 3 dimensions)
Running HDBSCAN with min_cluster_size = 50...
-> Found 7 clusters and 1007 noise points.

--- Statistical Profile of Clusters ---
          trans_x                   trans_y                   trans_z  \
             mean       std count      mean       std count      mean   
cluster                                                                 
0        0.327041  0.079258    59 -0.432603  0.053959    59 -0.250368   
1       -0.362030  0.071542   351 -0.474657  0.111666   351 -0.217186   
2       -0.392326  0.080605   331 -0.465684  0.065938   331  0.464437   
3        0.000425  0.013152   550 -0.137523  0.109399   550  0.122437   
4       -0.4412

/opt/homebrew/Caskroom/miniconda/base/envs/abs/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/abs/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


-> Successfully saved interactive plot to: singleton_cluster_plots/Translate_clusters.html

Analysis complete. Check the console for statistical profiles and the '{output_dir}' directory for interactive plots.


In [ ]:
debug_info("Starting singleton model training...")
singleton_models = find_abstractions(singleton_params, structure_type="SINGLETONS", min_examples=50, epochs=25)

debug_info("Starting pair model training...")
pair_models = find_abstractions(pair_params, structure_type="PAIRS", min_examples=50, epochs=25)

debug_success(f"Training complete. {len(singleton_models)} singleton models and {len(pair_models)} pair models trained.")

In [ ]:
if all_dsl_shapes and singleton_models and pair_models:
    random_chair = random.choice(all_dsl_shapes)
    debug_info("--- ORIGINAL CHAIR ---")
    debug_info(f"Serialized children count: {len(random_chair.serialize()[1][1])}")
    debug_info(f"Preview: {random_chair}")

    abstracted_chair = integrate_abstractions(
        random_chair, singleton_models, pair_models, error_threshold=0.01
    )

    debug_info("\n--- ABSTRACTED CHAIR ---")
    if isinstance(abstracted_chair, Abstraction):
        debug_info(f"Abstraction pattern: {abstracted_chair.pattern_name}, compressed dim: {len(abstracted_chair.compressed_params)}")
    debug_info(f"Preview: {abstracted_chair}")

    # Visualization
    plot_chair(random_chair)
    plot_chair(abstracted_chair)
else:
    debug_error("Cannot run test: DSL shapes or trained models are missing.")

In [ ]:
print("Original Chair Object:")
print(random_chair)

print("\nAbstracted Chair Object:")
print(abstracted_chair)